In [100]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/dgomonov/new-york-city-airbnb-open-data/AB_NYC_2019.csv
/kaggle/input/datasets/dgomonov/new-york-city-airbnb-open-data/New_York_City_.png


In [87]:
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, FunctionTransformer, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.svm import SVR
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

In [88]:
df = pd.read_csv("/kaggle/input/datasets/dgomonov/new-york-city-airbnb-open-data/AB_NYC_2019.csv")

In [89]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48895 entries, 0 to 48894
Data columns (total 16 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   id                              48895 non-null  int64  
 1   name                            48879 non-null  object 
 2   host_id                         48895 non-null  int64  
 3   host_name                       48874 non-null  object 
 4   neighbourhood_group             48895 non-null  object 
 5   neighbourhood                   48895 non-null  object 
 6   latitude                        48895 non-null  float64
 7   longitude                       48895 non-null  float64
 8   room_type                       48895 non-null  object 
 9   price                           48895 non-null  int64  
 10  minimum_nights                  48895 non-null  int64  
 11  number_of_reviews               48895 non-null  int64  
 12  last_review                     

In [90]:
df["room_type"].value_counts()

room_type
Entire home/apt    25409
Private room       22326
Shared room         1160
Name: count, dtype: int64

In [91]:
df = pd.get_dummies(df, columns=["room_type"], dtype=float) #one-hot encode

In [92]:
df = df.drop(columns=["host_name", "host_id", "id", "name"])
df.head(4)

,neighbourhood_group,neighbourhood,latitude,longitude,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365,room_type_Entire home/apt,room_type_Private room,room_type_Shared room
0,Brooklyn,Kensington,40.64749,-73.97237,149,1,9,2018-10-19,0.21,6,365,0.0,1.0,0.0
1,Manhattan,Midtown,40.75362,-73.98377,225,1,45,2019-05-21,0.38,2,355,1.0,0.0,0.0
2,Manhattan,Harlem,40.80902,-73.94190,150,3,0,NaN,NaN,1,365,0.0,1.0,0.0
3,Brooklyn,Clinton Hill,40.68514,-73.95976,89,1,270,2019-07-05,4.64,1,194,1.0,0.0,0.0


In [93]:
df = pd.get_dummies(df, columns = ["neighbourhood_group"], dtype = float)
df


,neighbourhood,latitude,longitude,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365,room_type_Entire home/apt,room_type_Private room,room_type_Shared room,neighbourhood_group_Bronx,neighbourhood_group_Brooklyn,neighbourhood_group_Manhattan,neighbourhood_group_Queens,neighbourhood_group_Staten Island
0,Kensington,40.64749,-73.97237,149,1,9,2018-10-19,0.21,6,365,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
1,Midtown,40.75362,-73.98377,225,1,45,2019-05-21,0.38,2,355,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
2,Harlem,40.80902,-73.94190,150,3,0,NaN,NaN,1,365,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
3,Clinton Hill,40.68514,-73.95976,89,1,270,2019-07-05,4.64,1,194,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
4,East Harlem,40.79851,-73.94399,80,10,9,2018-11-19,0.10,1,0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48890,Bedford-Stuyvesant,40.67853,-73.94995,70,2,0,NaN,NaN,2,9,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
48891,Bushwick,40.70184,-73.93317,40,4,0,NaN,NaN,2,36,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
48892,Harlem,40.81475,-73.94867,115,10,0,NaN,NaN,1,27,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
48893,Hell's Kitchen,40.75751,-73.99112,55,1,0,NaN,NaN,6,2,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0


In [94]:
#Ordinal encoding
enc = OrdinalEncoder()
df["neighbourhood"] = enc.fit_transform(df[["neighbourhood"]])
df.head(4)

,neighbourhood,latitude,longitude,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365,room_type_Entire home/apt,room_type_Private room,room_type_Shared room,neighbourhood_group_Bronx,neighbourhood_group_Brooklyn,neighbourhood_group_Manhattan,neighbourhood_group_Queens,neighbourhood_group_Staten Island
0,108.0,40.64749,-73.97237,149,1,9,2018-10-19,0.21,6,365,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
1,127.0,40.75362,-73.98377,225,1,45,2019-05-21,0.38,2,355,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
2,94.0,40.80902,-73.94190,150,3,0,NaN,NaN,1,365,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
3,41.0,40.68514,-73.95976,89,1,270,2019-07-05,4.64,1,194,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


In [95]:
df["last_review"] = pd.to_datetime(df["last_review"]).astype("int64")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48895 entries, 0 to 48894
Data columns (total 18 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   neighbourhood                      48895 non-null  float64
 1   latitude                           48895 non-null  float64
 2   longitude                          48895 non-null  float64
 3   price                              48895 non-null  int64  
 4   minimum_nights                     48895 non-null  int64  
 5   number_of_reviews                  48895 non-null  int64  
 6   last_review                        48895 non-null  int64  
 7   reviews_per_month                  38843 non-null  float64
 8   calculated_host_listings_count     48895 non-null  int64  
 9   availability_365                   48895 non-null  int64  
 10  room_type_Entire home/apt          48895 non-null  float64
 11  room_type_Private room             48895 non-null  flo

In [96]:
df = pd.read_csv("/kaggle/input/datasets/dgomonov/new-york-city-airbnb-open-data/AB_NYC_2019.csv")
df.head(4)

,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365
0,2539,Clean & quiet apt home by the park,2787,John,Brooklyn,Kensington,40.64749,-73.97237,Private room,149,1,9,2018-10-19,0.21,6,365
1,2595,Skylit Midtown Castle,2845,Jennifer,Manhattan,Midtown,40.75362,-73.98377,Entire home/apt,225,1,45,2019-05-21,0.38,2,355
2,3647,THE VILLAGE OF HARLEM....NEW YORK !,4632,Elisabeth,Manhattan,Harlem,40.80902,-73.94190,Private room,150,3,0,NaN,NaN,1,365
3,3831,Cozy Entire Floor of Brownstone,4869,LisaRoxanne,Brooklyn,Clinton Hill,40.68514,-73.95976,Entire home/apt,89,1,270,2019-07-05,4.64,1,194


In [97]:
df = df.drop(columns=["name", "host_id", "id", "host_name"])

X = df.drop(columns=["price"])
y = df["price"]

In [98]:
def date_string_to_float(date_cols : pd.DataFrame):
    dates = date_cols.apply(
        lambda col: pd.to_datetime(col, errors="coerce")
    )

    return (
        dates
        .sub(pd.Timestamp("2000-01-01"))
        .apply(lambda col: col.dt.days)
        .to_numpy(dtype=float)
    )

date_pipeline = Pipeline([
    (
        "converter",
        FunctionTransformer(
            date_string_to_float,
            feature_names_out=lambda _, features: [
                f"{features}_days" for feature in features
            ]
        )
    ),
    ("imputer", SimpleImputer(strategy="median"))
])

one_hot_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(sparse_output=False, handle_unknown="ignore"))
])

ordinal_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
])

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer()),
    ("scaler", StandardScaler())
])

preprocessor = ColumnTransformer(
    transformers=[
        (
            "one_hot",
            one_hot_pipeline,
            ["room_type", "neighbourhood_group"]
        ),
        (
            "ordinal",
            ordinal_pipeline,
            ["neighbourhood"]
        ),
        (
            "date",
            date_pipeline,
            ["last_review"]
        )
    ],
    remainder=numeric_pipeline
)

preprocessor

ColumnTransformer(remainder=Pipeline(steps=[('imputer', SimpleImputer()),
                                            ('scaler', StandardScaler())]),
                  transformers=[('one_hot',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False))]),
                                 ['room_type', 'neighbourhood_group']),
                                ('ordinal',
                                 Pipeline(steps=[('imputer',
                                                  SimpleI...y='most_frequent')),
                                                 ('encoder',
                                                  OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                 unknown_value=-1))]),
                                 ['neighbourhood']),
                                ('date',
                                 Pipeline(steps=[('converter',
                                                  FunctionTransformer(feature_names_out=<function <lambda> at 0x7af22efb3a60>,
                                                                      func=<function date_string_to_float at 0x7af22efb0040>)),
                                                 ('imputer',
                                                  SimpleImputer(strategy='median'))]),
                                 ['last_review'])])

In [99]:
pipe = Pipeline([
    ("preprocess", preprocessor),
    ("model", LinearRegression())
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)

pipe.fit(X_train, y_train)

y_pred = pipe.predict(X_test)

(y_pred - y_test)**2

32636    4313.417680
43862     465.168425
6044      385.971198
18053    2510.621656
4979     1584.709150
            ...     
36246     450.687357
31119     647.465105
24011      39.722809
6194      839.744770
45159    5517.734805
Name: price, Length: 14669, dtype: float64